In [1]:
import pandas as pd
import numpy as np

# update the path if needed
file_path = "Hackathon Data.csv"

df = pd.read_csv(file_path)

print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
print(df.head())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

if "type" in df.columns and "isFraud" in df.columns:
    print("\nTransaction types:")
    print(df["type"].value_counts())

    print("\nFraud counts:")
    print(df["isFraud"].value_counts())

    print("\nFraud by type:")
    print(df.groupby("type")["isFraud"].agg(["count", "sum", "mean"]))

Shape: (1048575, 10)

Columns:
['type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']

First 5 rows:
       type    amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0   PAYMENT   9839.64  C1231006815       170136.0       160296.36   
1   PAYMENT   1864.28  C1666544295        21249.0        19384.72   
2  TRANSFER    181.00  C1305486145          181.0            0.00   
3  CASH_OUT    181.00   C840083671          181.0            0.00   
4   PAYMENT  11668.14  C2048537720        41554.0        29885.86   

      nameDest  oldbalanceDest  newbalanceDest  isFraud  isFlaggedFraud  
0  M1979787155             0.0             0.0        0               0  
1  M2044282225             0.0             0.0        0               0  
2   C553264065             0.0             0.0        1               0  
3    C38997010         21182.0             0.0        1               0  
4  M1230701703            

In [2]:
# keep only columns for the first fraud modeling pipeline
df_model = df[["type", "amount", "nameOrig", "nameDest", "isFraud"]].copy()

print("Modeling shape:", df_model.shape)

print("\nModeling columns:")
print(df_model.columns.tolist())

print("\nMissing values in modeling data:")
print(df_model.isnull().sum())

print("\nFraud counts:")
print(df_model["isFraud"].value_counts())

print("\nFraud by type:")
print(df_model.groupby("type")["isFraud"].agg(["count", "sum", "mean"]))

Modeling shape: (1048575, 5)

Modeling columns:
['type', 'amount', 'nameOrig', 'nameDest', 'isFraud']

Missing values in modeling data:
type        0
amount      0
nameOrig    0
nameDest    0
isFraud     0
dtype: int64

Fraud counts:
isFraud
0    1047433
1       1142
Name: count, dtype: int64

Fraud by type:
           count  sum      mean
type                           
CASH_IN   227130    0  0.000000
CASH_OUT  373641  578  0.001547
DEBIT       7178    0  0.000000
PAYMENT   353873    0  0.000000
TRANSFER   86753  564  0.006501


In [3]:
print(df["isFlaggedFraud"].value_counts())

isFlaggedFraud
0    1048575
Name: count, dtype: int64


In [4]:
from sklearn.model_selection import train_test_split

# keep only transaction types where fraud actually exists
df_risky = df_model[df_model["type"].isin(["TRANSFER", "CASH_OUT"])].copy()

print("Risky shape:", df_risky.shape)

print("\nRisky fraud counts:")
print(df_risky["isFraud"].value_counts())

print("\nRisky fraud by type:")
print(df_risky.groupby("type")["isFraud"].agg(["count", "sum", "mean"]))

# split into train / temp
train_df, temp_df = train_test_split(
    df_risky,
    test_size=0.30,
    random_state=42,
    stratify=df_risky["isFraud"]
)

# split temp into validation / test
valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["isFraud"]
)

print("\nTrain shape:", train_df.shape)
print("Validation shape:", valid_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain fraud counts:")
print(train_df["isFraud"].value_counts())

print("\nValidation fraud counts:")
print(valid_df["isFraud"].value_counts())

print("\nTest fraud counts:")
print(test_df["isFraud"].value_counts())

print("\nTrain fraud rate:", train_df["isFraud"].mean())
print("Validation fraud rate:", valid_df["isFraud"].mean())
print("Test fraud rate:", test_df["isFraud"].mean())

Risky shape: (460394, 5)

Risky fraud counts:
isFraud
0    459252
1      1142
Name: count, dtype: int64

Risky fraud by type:
           count  sum      mean
type                           
CASH_OUT  373641  578  0.001547
TRANSFER   86753  564  0.006501

Train shape: (322275, 5)
Validation shape: (69059, 5)
Test shape: (69060, 5)

Train fraud counts:
isFraud
0    321476
1       799
Name: count, dtype: int64

Validation fraud counts:
isFraud
0    68888
1      171
Name: count, dtype: int64

Test fraud counts:
isFraud
0    68888
1      172
Name: count, dtype: int64

Train fraud rate: 0.0024792490885113647
Validation fraud rate: 0.0024761435873673237
Test fraud rate: 0.0024905878945844194


In [5]:
# rounded amount to avoid float-matching issues
for frame in [train_df, valid_df, test_df]:
    frame["amount_r"] = frame["amount"].round(2)

# ---------- TRAIN-ONLY lookup sets ----------
train_transfer_amounts = set(
    train_df.loc[train_df["type"] == "TRANSFER", "amount_r"]
)

train_cashout_amounts = set(
    train_df.loc[train_df["type"] == "CASH_OUT", "amount_r"]
)

paired_amounts_train = train_transfer_amounts.intersection(train_cashout_amounts)

dest_count_train = train_df["nameDest"].value_counts().to_dict()
amount_count_train = train_df["amount_r"].value_counts().to_dict()

print("Number of paired amounts in training:", len(paired_amounts_train))
print("Unique destinations in training:", len(dest_count_train))
print("Unique rounded amounts in training:", len(amount_count_train))

# ---------- apply rule features ----------
for frame in [train_df, valid_df, test_df]:
    frame["paired_amount_rule"] = frame["amount_r"].isin(paired_amounts_train).astype(int)
    frame["dest_count_train"] = frame["nameDest"].map(dest_count_train).fillna(0).astype(int)
    frame["amount_count_train"] = frame["amount_r"].map(amount_count_train).fillna(0).astype(int)

    frame["rare_dest_rule"] = (frame["dest_count_train"] <= 2).astype(int)
    frame["large_amount_rule"] = (frame["amount"] >= 3000000).astype(int)
    frame["repeated_amount_rule"] = (frame["amount_count_train"] >= 2).astype(int)

    # simple additive rule score
    frame["rule_score"] = (
        3 * frame["paired_amount_rule"] +
        2 * frame["rare_dest_rule"] +
        1 * frame["large_amount_rule"] +
        1 * frame["repeated_amount_rule"]
    )

print("\nValidation rule feature summary:")
print(valid_df[[
    "paired_amount_rule",
    "rare_dest_rule",
    "large_amount_rule",
    "repeated_amount_rule",
    "rule_score"
]].describe())

print("\nFraud rate by rule_score on validation:")
print(valid_df.groupby("rule_score")["isFraud"].agg(["count", "sum", "mean"]).sort_index())

print("\nFraud rate by rule_score on test:")
print(test_df.groupby("rule_score")["isFraud"].agg(["count", "sum", "mean"]).sort_index())

Number of paired amounts in training: 450
Unique destinations in training: 78453
Unique rounded amounts in training: 320931

Validation rule feature summary:
       paired_amount_rule  rare_dest_rule  large_amount_rule  \
count        69059.000000    69059.000000       69059.000000   
mean             0.000101        0.251307           0.001173   
std              0.010067        0.433768           0.034228   
min              0.000000        0.000000           0.000000   
25%              0.000000        0.000000           0.000000   
50%              0.000000        0.000000           0.000000   
75%              0.000000        1.000000           0.000000   
max              1.000000        1.000000           1.000000   

       repeated_amount_rule   rule_score  
count          69059.000000  69059.00000  
mean               0.000159      0.50425  
std                0.012620      0.86963  
min                0.000000      0.00000  
25%                0.000000      0.00000  
50%    

In [6]:
# hard rule = strong obvious fraud pattern
for frame in [train_df, valid_df, test_df]:
    frame["rule_block"] = (
        (frame["paired_amount_rule"] == 1) &
        (frame["rare_dest_rule"] == 1)
    ).astype(int)

print("Validation rule_block counts:")
print(valid_df["rule_block"].value_counts())

print("\nValidation fraud by rule_block:")
print(valid_df.groupby("rule_block")["isFraud"].agg(["count", "sum", "mean"]))

print("\nTest fraud by rule_block:")
print(test_df.groupby("rule_block")["isFraud"].agg(["count", "sum", "mean"]))

Validation rule_block counts:
rule_block
0    69054
1        5
Name: count, dtype: int64

Validation fraud by rule_block:
            count  sum      mean
rule_block                      
0           69054  168  0.002433
1               5    3  0.600000

Test fraud by rule_block:
            count  sum      mean
rule_block                      
0           69052  167  0.002418
1               8    5  0.625000


In [7]:
# keep only rows NOT hard-blocked by the rules
train_ml = train_df[train_df["rule_block"] == 0].copy()
valid_ml = valid_df[valid_df["rule_block"] == 0].copy()
test_ml  = test_df[test_df["rule_block"] == 0].copy()

print("Stage-2 train shape:", train_ml.shape)
print("Stage-2 valid shape:", valid_ml.shape)
print("Stage-2 test shape:", test_ml.shape)

print("\nStage-2 train fraud counts:")
print(train_ml["isFraud"].value_counts())

print("\nStage-2 valid fraud counts:")
print(valid_ml["isFraud"].value_counts())

print("\nStage-2 test fraud counts:")
print(test_ml["isFraud"].value_counts())

Stage-2 train shape: (321848, 14)
Stage-2 valid shape: (69054, 14)
Stage-2 test shape: (69052, 14)

Stage-2 train fraud counts:
isFraud
0    321393
1       455
Name: count, dtype: int64

Stage-2 valid fraud counts:
isFraud
0    68886
1      168
Name: count, dtype: int64

Stage-2 test fraud counts:
isFraud
0    68885
1      167
Name: count, dtype: int64


In [8]:
# TRAIN-only lookup tables
orig_count_train = train_ml["nameOrig"].value_counts().to_dict()
dest_count_train_ml = train_ml["nameDest"].value_counts().to_dict()

type_mean_amount = train_ml.groupby("type")["amount"].mean().to_dict()
type_median_amount = train_ml.groupby("type")["amount"].median().to_dict()

def add_ml_features(frame):
    frame = frame.copy()

    frame["orig_count_train"] = frame["nameOrig"].map(orig_count_train).fillna(0).astype(int)
    frame["dest_count_train_ml"] = frame["nameDest"].map(dest_count_train_ml).fillna(0).astype(int)

    frame["type_mean_amount"] = frame["type"].map(type_mean_amount)
    frame["type_median_amount"] = frame["type"].map(type_median_amount)

    frame["amount_to_type_mean"] = frame["amount"] / frame["type_mean_amount"]
    frame["amount_to_type_median"] = frame["amount"] / frame["type_median_amount"]

    frame["log_amount"] = np.log1p(frame["amount"])
    frame["log_orig_count"] = np.log1p(frame["orig_count_train"])
    frame["log_dest_count"] = np.log1p(frame["dest_count_train_ml"])
    frame["log_amount_count"] = np.log1p(frame["amount_count_train"])

    frame["is_transfer"] = (frame["type"] == "TRANSFER").astype(int)
    frame["is_cash_out"] = (frame["type"] == "CASH_OUT").astype(int)

    return frame

train_ml = add_ml_features(train_ml)
valid_ml = add_ml_features(valid_ml)
test_ml = add_ml_features(test_ml)

feature_cols = [
    "log_amount",
    "amount_to_type_mean",
    "amount_to_type_median",
    "log_orig_count",
    "log_dest_count",
    "log_amount_count",
    "paired_amount_rule",
    "rare_dest_rule",
    "large_amount_rule",
    "repeated_amount_rule",
    "rule_score",
    "is_transfer",
    "is_cash_out"
]

print("Feature columns:")
print(feature_cols)

print("\nMissing values in training features:")
print(train_ml[feature_cols].isnull().sum())

Feature columns:
['log_amount', 'amount_to_type_mean', 'amount_to_type_median', 'log_orig_count', 'log_dest_count', 'log_amount_count', 'paired_amount_rule', 'rare_dest_rule', 'large_amount_rule', 'repeated_amount_rule', 'rule_score', 'is_transfer', 'is_cash_out']

Missing values in training features:
log_amount               0
amount_to_type_mean      0
amount_to_type_median    0
log_orig_count           0
log_dest_count           0
log_amount_count         0
paired_amount_rule       0
rare_dest_rule           0
large_amount_rule        0
repeated_amount_rule     0
rule_score               0
is_transfer              0
is_cash_out              0
dtype: int64


In [9]:
# install once if needed
# !pip install xgboost

from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

X_train = train_ml[feature_cols]
y_train = train_ml["isFraud"]

X_valid = valid_ml[feature_cols]
y_valid = valid_ml["isFraud"]

X_test = test_ml[feature_cols]
y_test = test_ml["isFraud"]

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print("Negative class count:", neg)
print("Positive class count:", pos)
print("scale_pos_weight:", scale_pos_weight)

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

xgb_model.fit(X_train, y_train)

valid_ml["ml_score"] = xgb_model.predict_proba(X_valid)[:, 1]
test_ml["ml_score"] = xgb_model.predict_proba(X_test)[:, 1]

valid_ap = average_precision_score(y_valid, valid_ml["ml_score"])
valid_roc = roc_auc_score(y_valid, valid_ml["ml_score"])

print("\nValidation AP:", valid_ap)
print("Validation ROC-AUC:", valid_roc)

print("\nValidation score summary:")
print(valid_ml["ml_score"].describe())

feature_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": xgb_model.feature_importances_
}).sort_values("importance", ascending=False)

print("\nTop 10 feature importances:")
print(feature_importance.head(10))

Negative class count: 321393
Positive class count: 455
scale_pos_weight: 706.3582417582418

Validation AP: 0.22071302345645855
Validation ROC-AUC: 0.8471179695784476

Validation score summary:
count    69054.000000
mean         0.148530
std          0.182258
min          0.001814
25%          0.037346
50%          0.080843
75%          0.173679
max          0.997966
Name: ml_score, dtype: float64

Top 10 feature importances:
                  feature  importance
10             rule_score    0.449760
9    repeated_amount_rule    0.135367
5        log_amount_count    0.111661
11            is_transfer    0.062099
12            is_cash_out    0.055711
2   amount_to_type_median    0.040895
1     amount_to_type_mean    0.037158
0              log_amount    0.035054
4          log_dest_count    0.029793
7          rare_dest_rule    0.025577


In [10]:
print("Validation ML score by class:")
print(valid_ml.groupby("isFraud")["ml_score"].agg(["count", "mean", "median", "min", "max"]))

print("\nTest ML score by class:")
print(test_ml.groupby("isFraud")["ml_score"].agg(["count", "mean", "median", "min", "max"]))

Validation ML score by class:
         count      mean    median       min       max
isFraud                                               
0        68886  0.147399  0.080821  0.001814  0.997211
1          168  0.612178  0.780192  0.015908  0.997966

Test ML score by class:
         count      mean    median       min       max
isFraud                                               
0        68885  0.147305  0.081163  0.001814  0.997211
1          167  0.669252  0.791526  0.003807  0.999553


In [11]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score, roc_auc_score

# train anomaly model only on normal transactions
X_train_normal = train_ml.loc[train_ml["isFraud"] == 0, feature_cols].copy()

X_valid = valid_ml[feature_cols].copy()
X_test = test_ml[feature_cols].copy()

print("Normal training rows for anomaly model:", X_train_normal.shape)

iso_model = IsolationForest(
    n_estimators=300,
    contamination="auto",
    random_state=42,
    n_jobs=-1
)

iso_model.fit(X_train_normal)

# decision_function: smaller means more abnormal
valid_ml["iso_decision"] = iso_model.decision_function(X_valid)
test_ml["iso_decision"] = iso_model.decision_function(X_test)

# convert so bigger = more anomalous
valid_ml["anomaly_score"] = -valid_ml["iso_decision"]
test_ml["anomaly_score"] = -test_ml["iso_decision"]

print("\nValidation anomaly score summary:")
print(valid_ml["anomaly_score"].describe())

print("\nTest anomaly score summary:")
print(test_ml["anomaly_score"].describe())

print("\nValidation anomaly score by class:")
print(valid_ml.groupby("isFraud")["anomaly_score"].agg(["count", "mean", "median", "min", "max"]))

print("\nTest anomaly score by class:")
print(test_ml.groupby("isFraud")["anomaly_score"].agg(["count", "mean", "median", "min", "max"]))

Normal training rows for anomaly model: (321393, 13)

Validation anomaly score summary:
count    69054.000000
mean        -0.025615
std          0.076294
min         -0.117190
25%         -0.095342
50%         -0.042098
75%          0.023864
max          0.267382
Name: anomaly_score, dtype: float64

Test anomaly score summary:
count    69052.000000
mean        -0.025902
std          0.075911
min         -0.117190
25%         -0.095105
50%         -0.042209
75%          0.023718
max          0.279882
Name: anomaly_score, dtype: float64

Validation anomaly score by class:
         count      mean    median       min       max
isFraud                                               
0        68886 -0.025947 -0.042421 -0.117190  0.265135
1          168  0.110360  0.146126 -0.111439  0.267382

Test anomaly score by class:
         count      mean    median      min       max
isFraud                                              
0        68885 -0.026279 -0.042490 -0.11719  0.265653
1          

In [12]:
valid_ap_anom = average_precision_score(valid_ml["isFraud"], valid_ml["anomaly_score"])
valid_roc_anom = roc_auc_score(valid_ml["isFraud"], valid_ml["anomaly_score"])

test_ap_anom = average_precision_score(test_ml["isFraud"], test_ml["anomaly_score"])
test_roc_anom = roc_auc_score(test_ml["isFraud"], test_ml["anomaly_score"])

print("Validation AP (anomaly only):", valid_ap_anom)
print("Validation ROC-AUC (anomaly only):", valid_roc_anom)

print("\nTest AP (anomaly only):", test_ap_anom)
print("Test ROC-AUC (anomaly only):", test_roc_anom)

Validation AP (anomaly only): 0.09200896020197391
Validation ROC-AUC (anomaly only): 0.847440837380738

Test AP (anomaly only): 0.15344961740625318
Test ROC-AUC (anomaly only): 0.8789379939402606


In [19]:
# Step 7A + 7B together

orig_unique_dest_count = train_df.groupby("nameOrig")["nameDest"].nunique().to_dict()
dest_unique_orig_count = train_df.groupby("nameDest")["nameOrig"].nunique().to_dict()
pair_count_train = train_df.groupby(["nameOrig", "nameDest"]).size().to_dict()
orig_out_tx_count = train_df["nameOrig"].value_counts().to_dict()
dest_in_tx_count = train_df["nameDest"].value_counts().to_dict()

print("Unique origins in training:", train_df["nameOrig"].nunique())
print("Unique destinations in training:", train_df["nameDest"].nunique())
print("Unique origin-destination pairs in training:", len(pair_count_train))

def add_graph_features(frame):
    frame = frame.copy()

    frame["orig_unique_dest_count"] = frame["nameOrig"].map(orig_unique_dest_count).fillna(0).astype(int)
    frame["dest_unique_orig_count"] = frame["nameDest"].map(dest_unique_orig_count).fillna(0).astype(int)
    frame["orig_out_tx_count"] = frame["nameOrig"].map(orig_out_tx_count).fillna(0).astype(int)
    frame["dest_in_tx_count"] = frame["nameDest"].map(dest_in_tx_count).fillna(0).astype(int)

    frame["pair_count_train"] = [
        pair_count_train.get((o, d), 0)
        for o, d in zip(frame["nameOrig"], frame["nameDest"])
    ]

    frame["log_orig_unique_dest_count"] = np.log1p(frame["orig_unique_dest_count"])
    frame["log_dest_unique_orig_count"] = np.log1p(frame["dest_unique_orig_count"])
    frame["log_orig_out_tx_count"] = np.log1p(frame["orig_out_tx_count"])
    frame["log_dest_in_tx_count"] = np.log1p(frame["dest_in_tx_count"])
    frame["log_pair_count_train"] = np.log1p(frame["pair_count_train"])

    return frame

train_ml = add_graph_features(train_ml)
valid_ml = add_graph_features(valid_ml)
test_ml = add_graph_features(test_ml)

graph_feature_cols = [
    "log_orig_unique_dest_count",
    "log_dest_unique_orig_count",
    "log_orig_out_tx_count",
    "log_dest_in_tx_count",
    "log_pair_count_train"
]

print("\nMissing values in graph features (train):")
print(train_ml[graph_feature_cols].isnull().sum())

print("\nValidation graph features by class:")
print(valid_ml.groupby("isFraud")[graph_feature_cols].mean())

Unique origins in training: 322251
Unique destinations in training: 78453
Unique origin-destination pairs in training: 322275

Missing values in graph features (train):
log_orig_unique_dest_count    0
log_dest_unique_orig_count    0
log_orig_out_tx_count         0
log_dest_in_tx_count          0
log_pair_count_train          0
dtype: int64

Validation graph features by class:
         log_orig_unique_dest_count  log_dest_unique_orig_count  \
isFraud                                                           
0                          0.000171                    1.760984   
1                          0.000000                    0.981468   

         log_orig_out_tx_count  log_dest_in_tx_count  log_pair_count_train  
isFraud                                                                     
0                     0.000171              1.760984                   0.0  
1                     0.000000              0.981468                   0.0  


In [21]:
# create anomaly score for train_ml too
X_train_full = train_ml[feature_cols].copy()

train_ml["iso_decision"] = iso_model.decision_function(X_train_full)
train_ml["anomaly_score"] = -train_ml["iso_decision"]

print("Train anomaly score summary:")
print(train_ml["anomaly_score"].describe())

print("\nTrain anomaly score by class:")
print(train_ml.groupby("isFraud")["anomaly_score"].agg(["count", "mean", "median", "min", "max"]))

Train anomaly score summary:
count    321848.000000
mean         -0.035956
std           0.072511
min          -0.117190
25%          -0.097935
50%          -0.063806
75%           0.014965
max           0.290994
Name: anomaly_score, dtype: float64

Train anomaly score by class:
          count      mean    median       min       max
isFraud                                                
0        321393 -0.036229 -0.063997 -0.117190  0.267382
1           455  0.156436  0.175166 -0.113144  0.290994


In [22]:
all_feature_cols = feature_cols + graph_feature_cols + ["anomaly_score"]

print("Total number of features:", len(all_feature_cols))
print(all_feature_cols)

print("\nMissing values in combined train features:")
print(train_ml[all_feature_cols].isnull().sum())

print("\nMissing values in combined validation features:")
print(valid_ml[all_feature_cols].isnull().sum())

print("\nMissing values in combined test features:")
print(test_ml[all_feature_cols].isnull().sum())

Total number of features: 19
['log_amount', 'amount_to_type_mean', 'amount_to_type_median', 'log_orig_count', 'log_dest_count', 'log_amount_count', 'paired_amount_rule', 'rare_dest_rule', 'large_amount_rule', 'repeated_amount_rule', 'rule_score', 'is_transfer', 'is_cash_out', 'log_orig_unique_dest_count', 'log_dest_unique_orig_count', 'log_orig_out_tx_count', 'log_dest_in_tx_count', 'log_pair_count_train', 'anomaly_score']

Missing values in combined train features:
log_amount                    0
amount_to_type_mean           0
amount_to_type_median         0
log_orig_count                0
log_dest_count                0
log_amount_count              0
paired_amount_rule            0
rare_dest_rule                0
large_amount_rule             0
repeated_amount_rule          0
rule_score                    0
is_transfer                   0
is_cash_out                   0
log_orig_unique_dest_count    0
log_dest_unique_orig_count    0
log_orig_out_tx_count         0
log_dest_in_tx_co

In [23]:
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

X_train_all = train_ml[all_feature_cols]
y_train_all = train_ml["isFraud"]

X_valid_all = valid_ml[all_feature_cols]
y_valid_all = valid_ml["isFraud"]

X_test_all = test_ml[all_feature_cols]
y_test_all = test_ml["isFraud"]

neg = (y_train_all == 0).sum()
pos = (y_train_all == 1).sum()
scale_pos_weight_all = neg / pos

print("scale_pos_weight_all:", scale_pos_weight_all)

xgb_all = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight_all,
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

xgb_all.fit(X_train_all, y_train_all)

valid_ml["final_ml_score"] = xgb_all.predict_proba(X_valid_all)[:, 1]
test_ml["final_ml_score"] = xgb_all.predict_proba(X_test_all)[:, 1]

valid_ap_all = average_precision_score(y_valid_all, valid_ml["final_ml_score"])
valid_roc_all = roc_auc_score(y_valid_all, valid_ml["final_ml_score"])

test_ap_all = average_precision_score(y_test_all, test_ml["final_ml_score"])
test_roc_all = roc_auc_score(y_test_all, test_ml["final_ml_score"])

print("Validation AP (rules + ML + anomaly + graph):", valid_ap_all)
print("Validation ROC-AUC (rules + ML + anomaly + graph):", valid_roc_all)

print("\nTest AP (rules + ML + anomaly + graph):", test_ap_all)
print("Test ROC-AUC (rules + ML + anomaly + graph):", test_roc_all)

feature_importance_all = pd.DataFrame({
    "feature": all_feature_cols,
    "importance": xgb_all.feature_importances_
}).sort_values("importance", ascending=False)

print("\nTop 15 feature importances:")
print(feature_importance_all.head(15))


scale_pos_weight_all: 706.3582417582418
Validation AP (rules + ML + anomaly + graph): 0.22812582783645513
Validation ROC-AUC (rules + ML + anomaly + graph): 0.8538780168891875

Test AP (rules + ML + anomaly + graph): 0.2401257004951527
Test ROC-AUC (rules + ML + anomaly + graph): 0.8931672982698318

Top 15 feature importances:
                       feature  importance
10                  rule_score    0.375788
18               anomaly_score    0.186230
12                 is_cash_out    0.069861
16        log_dest_in_tx_count    0.056119
2        amount_to_type_median    0.055934
14  log_dest_unique_orig_count    0.048783
1          amount_to_type_mean    0.043771
0                   log_amount    0.041071
11                 is_transfer    0.037711
4               log_dest_count    0.033202
6           paired_amount_rule    0.020630
7               rare_dest_rule    0.015740
5             log_amount_count    0.008838
8            large_amount_rule    0.005833
9         repeated_amount_

In [24]:
print("Validation final score by class:")
print(valid_ml.groupby("isFraud")["final_ml_score"].agg(["count", "mean", "median", "min", "max"]))

print("\nTest final score by class:")
print(test_ml.groupby("isFraud")["final_ml_score"].agg(["count", "mean", "median", "min", "max"]))

Validation final score by class:
         count      mean    median       min       max
isFraud                                               
0        68886  0.134630  0.065196  0.001317  0.993899
1          168  0.621789  0.822906  0.004011  0.997066

Test final score by class:
         count      mean    median       min       max
isFraud                                               
0        68885  0.134100  0.065156  0.001317  0.993899
1          167  0.687702  0.862412  0.007118  0.998741


In [25]:
from sklearn.metrics import precision_score, recall_score, f1_score

high_thresholds = [0.97, 0.98, 0.99, 0.995]
low_thresholds  = [0.80, 0.85, 0.90, 0.95]

results = []

for high_thr in high_thresholds:
    for low_thr in low_thresholds:
        valid_actions = pd.Series("approve", index=valid_df.index, dtype="object")

        # Tier 1: hard rule block
        valid_actions.loc[valid_df["rule_block"] == 1] = "block"

        remaining_idx = valid_ml.index

        # block if very high model score
        block_idx = remaining_idx[
            valid_ml.loc[remaining_idx, "final_ml_score"] >= high_thr
        ]
        valid_actions.loc[block_idx] = "block"

        # review if not blocked and medium risk
        review_condition = (
            (valid_ml["final_ml_score"] >= low_thr) |
            (valid_ml["rule_score"] >= 3)
        )

        review_idx = remaining_idx[
            review_condition.loc[remaining_idx] &
            (valid_actions.loc[remaining_idx] != "block")
        ]
        valid_actions.loc[review_idx] = "review"

        y_true = valid_df["isFraud"]
        y_pred = valid_actions.isin(["block", "review"]).astype(int)

        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)

        block_count = (valid_actions == "block").sum()
        review_count = (valid_actions == "review").sum()
        approve_count = (valid_actions == "approve").sum()

        block_fraud = int(valid_df.loc[valid_actions == "block", "isFraud"].sum())
        review_fraud = int(valid_df.loc[valid_actions == "review", "isFraud"].sum())

        block_precision = block_fraud / block_count if block_count > 0 else 0
        review_precision = review_fraud / review_count if review_count > 0 else 0

        results.append({
            "high_thr": high_thr,
            "low_thr": low_thr,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "block_count": block_count,
            "review_count": review_count,
            "approve_count": approve_count,
            "block_fraud": block_fraud,
            "review_fraud": review_fraud,
            "block_precision": block_precision,
            "review_precision": review_precision
        })

threshold_df = pd.DataFrame(results).sort_values(
    ["f1", "precision", "recall"], ascending=False
)

print("Top threshold combinations:")
print(threshold_df.head(10))

Top threshold combinations:
    high_thr  low_thr  precision    recall        f1  block_count  \
3      0.970     0.95   0.305882  0.304094  0.304985           87   
7      0.980     0.95   0.305882  0.304094  0.304985           38   
11     0.990     0.95   0.305882  0.304094  0.304985           30   
15     0.995     0.95   0.305882  0.304094  0.304985           10   
2      0.970     0.90   0.092141  0.397661  0.149615           87   
6      0.980     0.90   0.092141  0.397661  0.149615           38   
10     0.990     0.90   0.092141  0.397661  0.149615           30   
14     0.995     0.90   0.092141  0.397661  0.149615           10   
1      0.970     0.85   0.070759  0.485380  0.123512           87   
5      0.980     0.85   0.070759  0.485380  0.123512           38   

    review_count  approve_count  block_fraud  review_fraud  block_precision  \
3             83          68889           39            13         0.448276   
7            132          68889           31          

In [26]:
usable_df = threshold_df[threshold_df["review_count"] <= 2000].sort_values(
    ["f1", "precision", "recall"], ascending=False
)

print("Top usable threshold combinations:")
print(usable_df.head(10))

best_row = usable_df.iloc[0]
print("\nBest row:")
print(best_row)

best_high = float(best_row["high_thr"])
best_low = float(best_row["low_thr"])

print("\nBest high threshold:", best_high)
print("Best low threshold:", best_low)

Top usable threshold combinations:
    high_thr  low_thr  precision    recall        f1  block_count  \
3      0.970     0.95   0.305882  0.304094  0.304985           87   
7      0.980     0.95   0.305882  0.304094  0.304985           38   
11     0.990     0.95   0.305882  0.304094  0.304985           30   
15     0.995     0.95   0.305882  0.304094  0.304985           10   
2      0.970     0.90   0.092141  0.397661  0.149615           87   
6      0.980     0.90   0.092141  0.397661  0.149615           38   
10     0.990     0.90   0.092141  0.397661  0.149615           30   
14     0.995     0.90   0.092141  0.397661  0.149615           10   
1      0.970     0.85   0.070759  0.485380  0.123512           87   
5      0.980     0.85   0.070759  0.485380  0.123512           38   

    review_count  approve_count  block_fraud  review_fraud  block_precision  \
3             83          68889           39            13         0.448276   
7            132          68889           31   

In [27]:
valid_actions = pd.Series("approve", index=valid_df.index, dtype="object")

valid_actions.loc[valid_df["rule_block"] == 1] = "block"

remaining_idx = valid_ml.index

block_idx = remaining_idx[
    valid_ml.loc[remaining_idx, "final_ml_score"] >= best_high
]
valid_actions.loc[block_idx] = "block"

review_condition = (
    (valid_ml["final_ml_score"] >= best_low) |
    (valid_ml["rule_score"] >= 3)
)

review_idx = remaining_idx[
    review_condition.loc[remaining_idx] &
    (valid_actions.loc[remaining_idx] != "block")
]
valid_actions.loc[review_idx] = "review"

print("Validation action counts:")
print(valid_actions.value_counts())

print("\nFraud rate by action:")
print(valid_df.groupby(valid_actions)["isFraud"].agg(["count", "sum", "mean"]))

Validation action counts:
approve    68889
block         87
review        83
Name: count, dtype: int64

Fraud rate by action:
         count  sum      mean
approve  68889  119  0.001727
block       87   39  0.448276
review      83   13  0.156627


In [29]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

# best thresholds from validation
best_high = 0.97
best_low = 0.95

# start everyone as approve
test_actions = pd.Series("approve", index=test_df.index, dtype="object")

# Tier 1: hard rule block
test_actions.loc[test_df["rule_block"] == 1] = "block"

remaining_idx = test_ml.index

# block if very high final model score
block_idx = remaining_idx[
    test_ml.loc[remaining_idx, "final_ml_score"] >= best_high
]
test_actions.loc[block_idx] = "block"

# review if medium risk and not already blocked
review_condition = (
    (test_ml["final_ml_score"] >= best_low) |
    (test_ml["rule_score"] >= 3)
)

review_idx = remaining_idx[
    review_condition.loc[remaining_idx] &
    (test_actions.loc[remaining_idx] != "block")
]
test_actions.loc[review_idx] = "review"

# treat block/review as positive fraud action
y_true_test = test_df["isFraud"]
y_pred_test = test_actions.isin(["block", "review"]).astype(int)

precision_test = precision_score(y_true_test, y_pred_test, zero_division=0)
recall_test = recall_score(y_true_test, y_pred_test, zero_division=0)
f1_test = f1_score(y_true_test, y_pred_test, zero_division=0)

print("Test Precision:", precision_test)
print("Test Recall:", recall_test)
print("Test F1:", f1_test)

print("\nTest action counts:")
print(test_actions.value_counts())

print("\nFraud rate by action on test:")
print(test_df.groupby(test_actions)["isFraud"].agg(["count", "sum", "mean"]))

print("\nConfusion matrix:")
print(confusion_matrix(y_true_test, y_pred_test))

Test Precision: 0.2864583333333333
Test Recall: 0.31976744186046513
Test F1: 0.3021978021978022

Test action counts:
approve    68868
review       101
block         91
Name: count, dtype: int64

Fraud rate by action on test:
         count  sum      mean
approve  68868  117  0.001699
block       91   40  0.439560
review     101   15  0.148515

Confusion matrix:
[[68751   137]
 [  117    55]]


In [30]:
summary_df = pd.DataFrame([{
    "model": "rules + ML + anomaly + graph",
    "high_threshold": best_high,
    "low_threshold": best_low,
    "precision": precision_test,
    "recall": recall_test,
    "f1": f1_test,
    "block_count": (test_actions == "block").sum(),
    "review_count": (test_actions == "review").sum(),
    "approve_count": (test_actions == "approve").sum(),
    "block_fraud_rate": test_df.loc[test_actions == "block", "isFraud"].mean(),
    "review_fraud_rate": test_df.loc[test_actions == "review", "isFraud"].mean(),
    "approve_fraud_rate": test_df.loc[test_actions == "approve", "isFraud"].mean()
}])

print(summary_df)

                          model  high_threshold  low_threshold  precision  \
0  rules + ML + anomaly + graph            0.97           0.95   0.286458   

     recall        f1  block_count  review_count  approve_count  \
0  0.319767  0.302198           91           101          68868   

   block_fraud_rate  review_fraud_rate  approve_fraud_rate  
0           0.43956           0.148515            0.001699  
